<a href="https://colab.research.google.com/github/Rasya-ai-web/Machine-Learning-Lab/blob/main/ML_13.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd


data = pd.read_csv("/content/Dataset.csv")

print("Dataset loaded successfully!")
print("Shape:", data.shape)
print(data.head())


data = data.drop(["id", "Unnamed: 32"], axis=1)


data["diagnosis"] = data["diagnosis"].map({
    "M": 1,
    "B": 0
})

X = data.drop("diagnosis", axis=1)
y = data["diagnosis"]

print("\nFeatures shape:", X.shape)
print("Target shape:", y.shape)

Dataset loaded successfully!
Shape: (569, 33)
   id diagnosis  radius_mean  texture_mean  perimeter_mean  area_mean  \
0   1         M        17.99         10.38          122.80     1001.0   
1   2         M        20.57         17.77          132.90     1326.0   
2   3         M        19.69         21.25          130.00     1203.0   
3   4         M        11.42         20.38           77.58      386.1   
4   5         M        20.29         14.34          135.10     1297.0   

   smoothness_mean  compactness_mean  concavity_mean  concave points_mean  \
0          0.11840           0.27760          0.3001              0.14710   
1          0.08474           0.07864          0.0869              0.07017   
2          0.10960           0.15990          0.1974              0.12790   
3          0.14250           0.28390          0.2414              0.10520   
4          0.10030           0.13280          0.1980              0.10430   

   ...  texture_worst  perimeter_worst  area_worst  

In [3]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Training data: (455, 30)
Testing data: (114, 30)


In [4]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=1000))
])

print("Model created successfully!")

Model created successfully!


In [5]:
model.fit(X_train, y_train)

print("Model trained successfully!")

Model trained successfully!


In [6]:
from sklearn.metrics import accuracy_score, classification_report


y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.9736842105263158

Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.99      0.98        71
           1       0.98      0.95      0.96        43

    accuracy                           0.97       114
   macro avg       0.97      0.97      0.97       114
weighted avg       0.97      0.97      0.97       114



In [7]:
import joblib

joblib.dump(model, "/content/model.pkl")

print("Model trained and saved successfully!")

Model trained and saved successfully!


In [8]:
%%writefile /content/app.py

from flask import Flask, request, jsonify
import joblib

# Create Flask application
app = Flask(__name__)

# Load trained model
model = joblib.load("/content/model.pkl")


@app.route("/")
def home():
    return jsonify({
        "message": "ML Model API is running!"
    })


@app.route("/predict", methods=["POST"])
def predict():

    try:
        # Receive JSON data
        data = request.get_json()

        # Get features
        features = data["features"]

        # Check number of features
        if len(features) != 30:
            return jsonify({
                "error": "Exactly 30 features are required."
            }), 400

        # Make prediction
        prediction = model.predict([features])[0]

        # Convert prediction to result
        if prediction == 1:
            result = "Malignant"
        else:
            result = "Benign"

        # Return JSON response
        return jsonify({
            "prediction": int(prediction),
            "result": result
        })

    except Exception as e:

        return jsonify({
            "error": str(e)
        }), 400


if __name__ == "__main__":
    app.run(
        host="0.0.0.0",
        port=5000,
        debug=False
    )

Writing /content/app.py


In [9]:
!pip install flask requests -q

print("Required libraries installed successfully!")

Required libraries installed successfully!


In [10]:
import threading
import time

from app import app

def run_server():
    app.run(
        host="0.0.0.0",
        port=5001,
        debug=False,
        use_reloader=False
    )

thread = threading.Thread(
    target=run_server,
    daemon=True
)

thread.start()

time.sleep(3)

print("Flask server started successfully!")
print("Running on port 5001")

 * Serving Flask app 'app'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5001
 * Running on http://172.28.0.12:5001
INFO:werkzeug:Press CTRL+C to quit


Flask server started successfully!
Running on port 5001


In [11]:
import requests

data = {
    "features": [
        17.99, 10.38, 122.8, 1001.0, 0.1184,
        0.2776, 0.3001, 0.1471, 0.2419, 0.07871,
        1.095, 0.9053, 8.589, 153.4, 0.006399,
        0.04904, 0.05373, 0.01587, 0.03003, 0.006193,
        25.38, 17.33, 184.6, 2019.0, 0.1622,
        0.6656, 0.7119, 0.2654, 0.4601, 0.1189
    ]
}

response = requests.post(
    "http://127.0.0.1:5001/predict",
    json=data
)

print("Status Code:", response.status_code)
print("Response:", response.json())

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
INFO:werkzeug:127.0.0.1 - - [05/Sep/2026 11:38:40] "POST /predict HTTP/1.1" 200 -


Status Code: 200
Response: {'prediction': 1, 'result': 'Malignant'}
